[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C09_Reasoning_TTC_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境自检

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy / matplotlib / pandas + 标准库，无任何其他依赖，所有 cell 秒级跑完。

**本 notebook 你将完成：**

1. **环境自检**：用优雅回退的 `check_import` 检查 numpy / matplotlib / pandas（缺失只标红、不报错）；
2. **热身 A · 模拟推理器雏形**：实现全课通用的 k 步推理链模型（每步正确率 p，可选自我纠错 r），蒙特卡洛验证解析式 $p^k$ 与 $(p+(1-p)r)^k$；
3. **热身 B · 误差复利热图**：画 P(成功) 随步数 k 与单步正确率 p 的热图，直观感受误差复利；
4. **热身 C · 多数投票的第一次威力**：采样 N 条链取多数，看到准确率怎么被"买"上去——以及错误集中时它如何失灵；
5. 完成 3 道 ✏️ 练习：`simulate_chain`（蒙特卡洛 vs 解析式）、`majority_vote`（含平票处理）、`pass_at_k`（无偏估计量）。

参考：[Wei 2022] *Chain-of-Thought Prompting* (arXiv:2201.11903)、[Wang 2022] *Self-Consistency* (arXiv:2203.11171)、[Brown 2024] *Large Language Monkeys* (arXiv:2407.21787)、[Snell 2024] *Scaling LLM Test-Time Compute Optimally* (arXiv:2408.03314)、[Chen 2021] *Codex / pass@k* (arXiv:2107.03374)、[DeepSeek-AI 2025] *DeepSeek-R1* (arXiv:2501.12948)。

## 1 · 环境自检：`check_import` 的优雅回退

本课全程只需要 numpy / matplotlib / pandas。`check_import` 是评测工程的标准模式：**依赖检查永远不应该让程序崩溃**——缺什么、缺了要不要紧，打印清楚就好。这个模式在后面模块（以及你自己的 eval harness）里会反复出现。

In [ ]:
import sys
import importlib

def check_import(name, required=True):
    """尝试导入模块并报告版本；失败时优雅回退（打印提示，绝不抛异常）。"""
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "unknown")
        print(f"  ✅ {name:<12s} {ver}")
        return mod
    except ImportError:
        if required:
            print(f"  ❌ {name:<12s} 未安装 —— 本课必需，请 pip install {name}")
        else:
            print(f"  ⚪ {name:<12s} 未安装 —— 本课刻意不依赖它，缺了完全正常")
        return None

print(f"Python {sys.version.split()[0]}  ({sys.platform})\n")

print("本课必需：")
for pkg in ["numpy", "matplotlib", "pandas"]:
    check_import(pkg, required=True)

print("\n本课刻意不依赖（装没装都行，全课纯 numpy 模拟推理器，无需 API key）：")
for pkg in ["torch", "scipy", "sklearn", "openai"]:
    check_import(pkg, required=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

rng = np.random.default_rng(0)   # 全课随机数纪律：一律 np.random.default_rng(0)，所有实验可复现

# numpy 冒烟测试：向量化采样
flips = rng.random((1000, 8)) < 0.95
assert flips.shape == (1000, 8) and 0.90 < flips.mean() < 1.0

# pandas 冒烟测试：p^k 速览表 —— 先感受一下误差复利的数量级
df = pd.DataFrame({"k": [1, 5, 10, 20, 40]})
for p in [0.99, 0.95, 0.90]:
    df[f"p={p}"] = p ** df["k"].to_numpy(dtype=float)
print(df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# matplotlib 冒烟测试：能出图即可
ks = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(5.5, 3))
for p in [0.99, 0.95, 0.90]:
    ax.plot(ks, p ** ks.astype(float), label=f"p={p}")
ax.set_xlabel("k steps"); ax.set_ylabel("$p^k$")
ax.set_title("smoke test: error compounding")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\n✅ numpy / pandas / matplotlib 全部就绪")

## 2 · 热身 A：模拟推理器雏形 —— k 步链与 $p^k$

全课的"理想气体模型"：**一条推理链 = k 个步骤，每步独立以概率 p 做对；最终答对当且仅当全部步骤正确。**于是

$$P_{\text{chain}} = p^k,\qquad \text{带自我纠错（做错的步以概率 } r \text{ 被救回）：} P = \big(p + (1-p)\,r\big)^k$$

下面用蒙特卡洛验证解析式：模拟 10 万条链，估计值与真值之差应落在 ~3 个二项标准误 $\sqrt{P(1-P)/n}$ 之内。这一步看似平凡，却是全课的**方法论宣言**：每个模拟实验都要（在能算的地方）与解析式对账——先校准仪器，再去测现象。

In [ ]:
def chain_success_mc(p, k, n_chains, rng, r=0.0):
    """模拟 n_chains 条 k 步推理链，返回成功率（全部步骤最终正确的比例）。
    每步以概率 p 直接做对；做错的步以概率 r 被自我纠错救回。"""
    ok = rng.random((n_chains, k)) < p              # 每步是否直接做对
    if r > 0:
        rescued = rng.random((n_chains, k)) < r
        ok = ok | (~ok & rescued)                   # 错了但被纠错救回
    return ok.all(axis=1).mean()

def chain_success_exact(p, k, r=0.0):
    q = p + (1 - p) * r                              # 单步"最终正确"的有效概率
    return q ** k

N_CHAINS = 100_000
rows = []
for p, k, r in [(0.99, 10, 0.0), (0.95, 10, 0.0), (0.95, 20, 0.0),
                (0.90, 20, 0.0), (0.90, 20, 0.5), (0.80, 30, 0.7)]:
    mc = chain_success_mc(p, k, N_CHAINS, rng, r)
    ex = chain_success_exact(p, k, r)
    se = np.sqrt(ex * (1 - ex) / N_CHAINS)           # 二项标准误
    rows.append({"p": p, "k": k, "r": r, "MC估计": mc, "解析式": ex,
                 "|误差|": abs(mc - ex), "3·SE": 3 * se})
table = pd.DataFrame(rows)
print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
assert (table["|误差|"] < table["3·SE"] + 1e-12).all()

print("\n✅ 蒙特卡洛与解析式在 3 个标准误内一致 —— 仪器校准完成")
print(f"注意第 3 行：每步 95% 正确的 20 步链，成功率只剩 {0.95**20:.1%}；")
print(f"而给同样的链加上 r=0.5 的自我纠错（第 5 行 p=0.90），成功率从 {0.90**20:.1%} 回到 {0.95**20:.1%}")
print("—— 反思/自查行为在数学上等价于抬高有效单步正确率，这是 o1/R1 长 CoT 的统计学意义之一。")

## 3 · 热身 B：误差复利热图 —— P(成功) vs (p, k)

把 $p^k$ 整个铺开：横轴单步正确率 p、纵轴步数 k、颜色 = 链成功率。白色等高线是 $P=0.5$，即 $k = \ln 0.5 / \ln p$——注意它弯得多陡：p 从 0.95 提到 0.99，"还能保住一半成功率"的链长就从 ~13 步暴涨到 ~69 步。**单步质量的微小提升在长链上被指数放大**——这就是为什么 frontier 实验室肯花大钱把单步错误率再压低一个百分点。

In [ ]:
ps = np.linspace(0.80, 0.999, 121)               # 单步正确率
ks = np.arange(1, 41)                            # 链长（步数）
P = ps[None, :] ** ks[:, None].astype(float)     # P[i, j] = ps[j] ** ks[i]

fig, ax = plt.subplots(figsize=(8, 4.6))
im = ax.pcolormesh(ps, ks, P, cmap="viridis", vmin=0, vmax=1, shading="auto")
cs = ax.contour(ps, ks, P, levels=[0.5, 0.9], colors=["white", "red"], linewidths=1.3)
ax.clabel(cs, fmt={0.5: "P=0.5", 0.9: "P=0.9"}, fontsize=9)
ax.set_xlabel("per-step accuracy  p")
ax.set_ylabel("chain length  k")
ax.set_title("error compounding:  P(chain correct) = $p^k$")
fig.colorbar(im, ax=ax, label="P(chain correct)")
plt.tight_layout(); plt.show()

for p in [0.90, 0.95, 0.99]:
    k50 = np.log(0.5) / np.log(p)
    print(f"p = {p:.2f}: 链长超过 {k50:5.1f} 步，成功率就跌破 50%")
print("\n误差复利是推理模型一切设计的出发点：CoT 把任务拆成更多步（k 变大）、")
print("但每步更简单（p 变大）—— 两个方向拔河谁赢？模块 01 的主线问题。")

## 4 · 热身 C：采样 N 次取多数 —— 第一次见识并行 TTC 的威力

单链成功率 q 不够高没关系：**采样 N 条独立链，对最终答案做 majority vote**（self-consistency, [Wang 2022]）。若 q > 0.5 且链间独立，多数票正确率随 N 升向 1（Condorcet 陪审团定理；N 取奇数避免平票）：

$$P_{\text{maj}}(N) = \sum_{j > N/2} \binom{N}{j}\, q^j (1-q)^{N-j}$$

更妙的是：**即使 q < 0.5，只要错误分散在很多不同答案上**，正确答案仍可能是唯一的众数（plurality 投票照样赢）。但反过来，若大家都掉进同一个坑（错误集中），投票会放大错误。下面同时演示威力与失灵的边界。

In [ ]:
def sample_answers(q, N, n_trials, rng, n_wrong=9, concentrated=False):
    """模拟 n_trials 道题、每题独立采样 N 个最终答案。正确答案记 0；
    答错时：concentrated=False 从 n_wrong 个错误答案中均匀抽一个（错得散），
            concentrated=True 全部押同一个错误答案 1（错得齐）。"""
    correct = rng.random((n_trials, N)) < q
    if concentrated:
        wrong = np.ones((n_trials, N), dtype=int)
    else:
        wrong = rng.integers(1, n_wrong + 1, size=(n_trials, N))
    return np.where(correct, 0, wrong)

def majority_acc(answers, n_vals=10):
    """逐题做 majority vote，返回众数 == 正确答案(0) 的比例。
    注：argmax 平票取最小 id，会偏袒答案 0 —— 演示够用；练习 2 写严格平票规则。"""
    counts = (answers[:, :, None] == np.arange(n_vals)[None, None, :]).sum(axis=1)
    return (counts.argmax(axis=1) == 0).mean()

Ns = np.arange(1, 42, 2)                          # 奇数 N，避开平票的主要影响
n_trials = 4000
configs = [(0.70, False, "q=0.70, scattered errors",        "-"),
           (0.55, False, "q=0.55, scattered errors",        "-"),
           (0.35, False, "q=0.35, scattered (1 of 9)",      "-"),
           (0.45, True,  "q=0.45, all on same wrong answer", "--")]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for q, conc, label, style in configs:
    accs = [majority_acc(sample_answers(q, N, n_trials, rng, concentrated=conc))
            for N in Ns]
    ax.plot(Ns, accs, style, marker=".", label=label)
ax.axhline(0.5, color="gray", lw=0.8, ls=":")
ax.set_xlabel("N (number of sampled chains)"); ax.set_ylabel("majority-vote accuracy")
ax.set_title("self-consistency: buying accuracy with samples")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("三个观察：")
print("① q>0.5 且错误分散：N 越大正确率单调升向 1（Condorcet）——准确率可以用算力买；")
print("② q=0.35 < 0.5 也能被救起：错误分散在 9 个答案上，正确答案仍是唯一众数（plurality）；")
print("③ 错误集中且 q<0.5：N 越大越『自信地错』——投票放大了系统性错误。")
print("   真实模型介于两端之间：答案间的相关性决定收益大小 —— 模块 02 定量展开。")

---
## ✏️ 练习 1：实现 `simulate_chain` —— 蒙特卡洛 vs 解析式

实现 `simulate_chain(p, k, rng, n_trials=20_000)`：模拟 `n_trials` 条 k 步推理链（每步独立以概率 p 正确，**无**自我纠错），返回"全部步骤正确"的比例（float）。这是全课模拟推理器的最小核心，模块 01 起会在它之上加自我纠错、难度分布与步骤相关性。

**提示**：核心一行——画一个 `(n_trials, k)` 的均匀随机矩阵与 p 比较，`all(axis=1)` 后取 `mean()`。注意 `rng.random()` 返回 $[0,1)$ 上的均匀数，所以 `< p` 在 `p=1.0` 时恒真、`p=0.0` 时恒假——两个边界因此是**精确**的 1.0 和 0.0。自测用 3.5 个二项标准误判定蒙特卡洛与 $p^k$ 一致。3–6 行。

In [ ]:
def simulate_chain(p, k, rng, n_trials=20_000):
    # TODO: 模拟 n_trials 条 k 步链（每步独立以概率 p 正确），
    #       返回全部步骤正确的比例（float）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng_t = np.random.default_rng(0)
for p, k in [(0.95, 20), (0.90, 10), (0.80, 5), (0.99, 50)]:
    est = simulate_chain(p, k, rng_t)
    true = p ** k
    se = np.sqrt(true * (1 - true) / 20_000)
    assert abs(est - true) < 3.5 * se + 1e-12, (p, k, est, true)   # 与解析式在置信区间内一致
assert simulate_chain(1.0, 30, rng_t) == 1.0       # 边界：每步必对 -> 链必对（精确）
assert simulate_chain(0.0, 3, rng_t) == 0.0        # 边界：每步必错 -> 链必错（精确）
assert 0.0 <= simulate_chain(0.5, 4, rng_t) <= 1.0
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `majority_vote` —— 含平票处理

实现 `majority_vote(answers)`：输入答案列表（元素可以是 int 或 str，顺序 = 采样顺序），返回出现次数最多的答案；**平票时返回平票者中在列表里最先出现的那个**（确定性规则，保证可复现——热身 C 里 `argmax` 的隐式规则偏袒小 id，这里做严格版）。空列表抛 `ValueError`。

**提示**：`collections.Counter` 数频次；再用一次遍历 + `dict.setdefault` 记录每个答案的首次出现位置 `first`；选择键 `max(counts, key=lambda a: (counts[a], -first[a]))`——次数多者优先，次数相同先到者优先。约 8 行。

In [ ]:
from collections import Counter

def majority_vote(answers):
    # TODO: 返回出现次数最多的答案；平票返回最先出现者；空列表 raise ValueError
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert majority_vote([1, 2, 1]) == 1                       # 简单多数
assert majority_vote(["B", "A", "B", "A"]) == "B"          # 平票：B 最先出现
assert majority_vote([2, 1, 1, 2]) == 2                    # 平票：2 最先出现
assert majority_vote([7]) == 7                             # 单元素
assert majority_vote(["x", "y", "y", "x", "y"]) == "y"
assert majority_vote([3, 3, 1, 1, 1]) == 1                 # 多数压倒"先出现"
try:
    majority_vote([])
    assert False, "空列表应抛 ValueError"
except ValueError:
    pass
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `pass_at_k` —— 无偏估计量

实现 `pass_at_k(n, c, k)`：一道题采样了 n 次、其中 c 次正确，返回 pass@k 的无偏估计 [Chen 2021, arXiv:2107.03374]

$$\widehat{\text{pass@}k} \;=\; 1 - \frac{\binom{n-c}{k}}{\binom{n}{k}}$$

直觉：从 n 个样本中**无放回**抽 k 个，"抽到的全是错的"的概率是 $\binom{n-c}{k}\big/\binom{n}{k}$。朴素做法（只采 k 个看有没有对的）方差大且分组平均后有偏；这个估计量把全部 n 个样本的信息都用上。LLM_Evals_Course 06 讲过它，这里亲手写——模块 02 的 coverage 曲线、模块 07 的报告口径全靠它。

**提示**：用 `math.comb`；先校验 `0 <= c <= n` 且 `1 <= k <= n`，违反抛 `ValueError`；再处理边界 `n - c < k`（错误样本不够填满 k 个抽样，必中）直接返回 `1.0`。约 8 行。

In [ ]:
from math import comb

def pass_at_k(n, c, k):
    # TODO: 无偏估计 1 - C(n-c, k) / C(n, k)
    #       参数不合法抛 ValueError；n-c < k 时返回 1.0
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
from itertools import combinations

assert pass_at_k(10, 0, 5) == 0.0                          # 边界：一次都没对
assert pass_at_k(10, 10, 1) == 1.0                         # 边界：全对
assert pass_at_k(10, 8, 5) == 1.0                          # 边界：n-c=2 < k=5，必中
assert abs(pass_at_k(10, 3, 1) - 0.3) < 1e-12              # pass@1 = c/n
assert abs(pass_at_k(5, 2, 2) - (1 - comb(3, 2) / comb(5, 2))) < 1e-12

# 与穷举对照：n=6, c=2，枚举所有 C(6,k) 个无放回子集，"含正确样本"的比例必须等于公式
outcomes = [1, 1, 0, 0, 0, 0]
for k in range(1, 7):
    brute = np.mean([max(sub) for sub in combinations(outcomes, k)])
    assert abs(pass_at_k(6, 2, k) - brute) < 1e-12, k

vals = [pass_at_k(20, 5, k) for k in range(1, 21)]
assert all(vals[i] <= vals[i + 1] + 1e-12 for i in range(len(vals) - 1))   # 对 k 单调不减

try:
    pass_at_k(5, 6, 1)
    assert False, "c > n 应抛 ValueError"
except ValueError:
    pass
print("✅ 练习 3 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def simulate_chain(p, k, rng, n_trials=20_000):
    steps_ok = rng.random((n_trials, k)) < p
    return float(steps_ok.all(axis=1).mean())

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
from collections import Counter

def majority_vote(answers):
    if not answers:
        raise ValueError("answers 不能为空")
    counts = Counter(answers)
    first = {}
    for i, a in enumerate(answers):
        first.setdefault(a, i)
    return max(counts, key=lambda a: (counts[a], -first[a]))

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
from math import comb

def pass_at_k(n, c, k):
    if not (0 <= c <= n and 1 <= k <= n):
        raise ValueError("需要 0 <= c <= n 且 1 <= k <= n")
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

---
## 小结

- **误差复利**：独立步骤模型下 $P_{\text{chain}} = p^k$——每步 95% 正确的 20 步链只剩 ~36%；自我纠错在数学上等价于抬高有效单步正确率 $q = p + (1-p)r$。这是推理模型一切设计（CoT、采样、verifier、搜索）要对抗的基线事实（模块 01）。
- **采样聚合**：majority vote 在"错误分散、采样独立"时把准确率买向 1；错误集中时 N 越大越自信地错——收益由答案间相关性决定（模块 02）。
- **测量三件套已就位**：`simulate_chain`（数据生成）、`majority_vote`（聚合）、`pass_at_k`（无偏估计）会在模块 01–07 被反复复用，请保管好。
- 全课方法论：在 ground truth 完全可控的**模拟推理器**上做统计实验——先与解析式对账校准仪器，再去测现象；外推到真实模型时记住"步骤独立"假设系统性偏乐观。

**下一步 → 模块 01 · CoT 与任务分解：误差复利的数学**——把"拆步骤让 p 变大、k 变多"这场拔河做成定量模型：什么时候 CoT 是净收益、什么时候分解反而有害。